In [10]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import yaml
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve().parent
with open(ROOT/'config.yaml','r') as file:
    config = yaml.safe_load(file)

df = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_usable = df[df.usable]

meta_cols = ['participant', 'epoch_number', 'label', 'usable', 'reason', 'target']
feature_cols = [c for c in df_usable.columns if c not in meta_cols]

X = df_usable[feature_cols].values
y = df_usable.label.values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=config['seed'])
groups = df_usable['participant'].values
folds = list(sgkf.split(X,y,groups))
fold_mapping = []
for fold_idx, (train_idx, val_idx) in enumerate(folds):
    val_participants = np.unique(groups[val_idx])
    for p in val_participants:
        fold_mapping.append({'participant': p, 'fold': fold_idx})

df_folds = pd.DataFrame(fold_mapping)
df_folds.to_csv(ROOT/'results'/'folds.csv')

print('Saved participant to fold list in results/folds.csv')

Saved participant to fold list in results/folds.csv


In [8]:
from sklearn.metrics import f1_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, precision_recall_fscore_support

def compute_metrics(y_true, y_pred, classes):
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=classes)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred, labels=classes)
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=classes, zero_division=0)
    stage_metrics = pd.DataFrame({'Stage': classes,'Precision': prec,'Recall': rec,'F1-Score': f1})
    
    return {'macro_f1': macro_f1,'balanced_accuracy': bal_acc,'kappa': kappa,'stage_metrics': stage_metrics,'confusion_matrix': cm}

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def run_baselines(X, y, model_name, model_fn, classes, folds, scale=False):
    fold_macro_f1s = []
    fold_bal_accs = []
    fold_kappas = []
    accumulated_cm = np.zeros((len(classes), len(classes)), dtype=int)

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        if scale:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

        if model_name == 'majority class':
            maj = pd.Series(y_train).mode()[0]
            preds = np.array([maj]*len(y_val))
        else:
            model = model_fn()
            model.fit(X_train,y_train)
            preds = model.predict(X_val)

        metrics = compute_metrics(y_val, preds, classes)
        fold_macro_f1s.append(metrics['macro_f1'])
        fold_bal_accs.append(metrics['balanced_accuracy'])
        fold_kappas.append(metrics['kappa'])
        accumulated_cm += metrics['confusion_matrix']

    print(f'{model_name.upper():}')
    print(f"\tMacro-F1: {np.mean(fold_macro_f1s):.4f} ± {np.std(fold_macro_f1s):.4f}")
    print(f"\tBalanced Accuracy: {np.mean(fold_bal_accs):.4f}")
    print(f"\tCohen's Kappa: {np.mean(fold_kappas):.4f}")

classes = np.array(['W', 'N1', 'N2', 'N3', 'R'])
run_baselines(X,y,'majority class',None,classes,folds=folds,scale=True)
run_baselines(X,y,'logistic regression',
              lambda: LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
              classes,folds,scale=True)
run_baselines(X,y,'random forest', 
             lambda: RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42),
             classes, folds)

MAJORITY CLASS
	Macro-F1: 0.1332 ± 0.0014
	Balanced Accuracy: 0.2000
	Cohen's Kappa: 0.0000
LOGISTIC REGRESSION
	Macro-F1: 0.2967 ± 0.0133
	Balanced Accuracy: 0.3980
	Cohen's Kappa: 0.1702
RANDOM FOREST
	Macro-F1: 0.3188 ± 0.0078
	Balanced Accuracy: 0.3255
	Cohen's Kappa: 0.2798
